# Qwen MLP-neuron prompt playground

Test hypotheses about a layer-16 `down_proj` neuron on arbitrary prompts. Token backgrounds show the raw signed coefficient entering `down_proj`: blue is positive, red is negative, and intensity is a fixed linear function of magnitude (saturating at `|score| = 25`). Hover over a token for its index, token ID, and exact score.

In [1]:
from __future__ import annotations

import html
import json
from pathlib import Path

import torch
from gb10_load_llm import load_model_to_cuda
from IPython.display import HTML, display
from transformers import AutoModelForCausalLM, AutoTokenizer

from icalens._capture import transformer_blocks

ROOT = Path.cwd()
if not (ROOT / 'local-icalens-models').is_dir():
    # Also works when Jupyter starts in this experiment directory.
    ROOT = Path('../..').resolve()
RUN = ROOT / 'experiments/mlp-neuron-kurtosis-profiles/runs/main/run.json'
resolved = json.loads(RUN.read_text())['resolved']
MODEL_ID = resolved['model']['repo_id']
REVISION = resolved['model']['revision']
LAYER = int(resolved['layer'])
FRAMING_TOKEN_ID = int(resolved['document_framing']['token_id'])
# Choose once for every analysis below:
#   'model'    = tokenizer's natural/default special-token behavior
#   'recorded' = prepend the <|endoftext|> used by the fitting/profile run
#   'none'     = encode raw text with no special tokens
DOCUMENT_FRAMING = 'model'

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=REVISION, trust_remote_code=True
)
model = load_model_to_cuda(
    AutoModelForCausalLM,
    MODEL_ID,
    revision=REVISION,
    device='cuda',
    dtype=torch.bfloat16,
    touch='auto',
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.eval()
down_proj = transformer_blocks(model)[LAYER].mlp.down_proj
print(f'Loaded {MODEL_ID} · layer {LAYER} · {down_proj.weight.shape[1]:,} neurons')

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Loaded Qwen/Qwen3.5-9B-Base · layer 16 · 12,288 neurons


In [2]:
class _CaptureComplete(Exception):
    pass


def neuron_scores(prompt: str, neuron: int = 11960):
    """Return prompt-token texts and raw post-gating MLP coefficients."""
    width = int(down_proj.weight.shape[1])
    if not 0 <= neuron < width:
        raise ValueError(f'neuron must be in 0..{width - 1}')
    if DOCUMENT_FRAMING not in {'model', 'recorded', 'none'}:
        raise ValueError("DOCUMENT_FRAMING must be 'model', 'recorded', or 'none'")
    content_ids = tokenizer(
        prompt, add_special_tokens=DOCUMENT_FRAMING == 'model'
    )['input_ids']
    prefix = [FRAMING_TOKEN_ID] if DOCUMENT_FRAMING == 'recorded' else []
    input_ids = torch.tensor([prefix + content_ids], dtype=torch.long, device='cuda')
    captured = None

    def hook(_, inputs):
        nonlocal captured
        captured = inputs[0][0, :, neuron].detach().float().cpu()
        raise _CaptureComplete

    handle = down_proj.register_forward_pre_hook(hook)
    try:
        with torch.inference_mode():
            try:
                model(input_ids=input_ids, attention_mask=torch.ones_like(input_ids), use_cache=False)
            except _CaptureComplete:
                pass
    finally:
        handle.remove()
    if captured is None:
        raise RuntimeError('down_proj hook did not run')
    offset = len(prefix)
    return [
        {
            'index': index,
            'token_id': token_id,
            'token': tokenizer.convert_ids_to_tokens(token_id),
            'text': tokenizer.decode([token_id], clean_up_tokenization_spaces=False),
            'score': float(captured[index + offset]),
        }
        for index, token_id in enumerate(content_ids)
    ]


def analyze(
    prompt: str,
    neuron: int = 11960,
    *,
    underline=None,
    color_cap: float = 25.0,
):
    """Score and display a prompt. `underline` may be an index or collection of indices."""
    if color_cap <= 0:
        raise ValueError('color_cap must be positive')
    rows = neuron_scores(prompt, neuron)
    if underline is None:
        underlined = set()
    elif isinstance(underline, int):
        underlined = {underline % len(rows)}
    else:
        underlined = {int(index) % len(rows) for index in underline}
    spans = []
    for row in rows:
        score = row['score']
        alpha = 0.02 + 0.56 * min(abs(score) / color_cap, 1.0)
        rgb = '61,95,153' if score >= 0 else '180,95,77'
        decoration = 'text-decoration:underline 1.5px;text-underline-offset:3px;' if row['index'] in underlined else ''
        title = html.escape(
            f"index {row['index']} · token {row['token']!r} · id {row['token_id']} · score {score:+.4f}",
            quote=True,
        )
        spans.append(
            f"<span title='{title}' style='background:rgba({rgb},{alpha:.3f});"
            f"border-radius:2px;{decoration}'>{html.escape(row['text'])}</span>"
        )
    strongest = sorted(rows, key=lambda row: abs(row['score']), reverse=True)[:5]
    summary = ' · '.join(f"#{row['index']} {row['score']:+.3f}" for row in strongest)
    display(HTML(
        f"<div style='font:14px/1.7 ui-monospace,SFMono-Regular,Menlo,monospace;"
        f"white-space:pre-wrap;border:1px solid #d8dee8;border-radius:9px;padding:15px'>"
        f"<div style='font:12px ui-sans-serif,system-ui;margin-bottom:10px;color:#667085'>"
        f"L{LAYER} N{neuron} · framing {DOCUMENT_FRAMING} · blue positive · red negative · linear cap ±{color_cap:g}<br>"
        f"strongest: {summary}</div>{''.join(spans)}</div>"
    ))
    return rows


def show_token_table(rows, minimum_absolute_score: float = 0.0):
    """Print token indices and exact values for constructing controlled tests."""
    for row in rows:
        if abs(row['score']) >= minimum_absolute_score:
            print(f"{row['index']:4d}  {row['score']:+9.4f}  {row['text']!r}")

## Edit and rerun this cell

`underline` is optional. It accepts one zero-based token index, a negative index counted from the end, or a list of indices. The returned `rows` contain exact token indices and scores.

In [28]:
prompt = '''
Prison sentences of up to 30 years are possible. On the island of Zanzibar, which belongs to Tanzania, lesbian sex is also punishable by five years in prison. In the rest of the country, however, there is no law against women.
'''

rows = analyze(prompt, neuron=10492)

In [29]:
# Optional: inspect exact values and token indices.
show_token_table(rows, minimum_absolute_score=0.5)

  26    -1.0625  ','
  43    -1.8125  ' country'
  44    -0.9727  ','
  45    -0.6719  ' however'
  46    -0.9336  ','
